# 01 - Data feasibility exploration

This notebook inspects the reviewed public API capture offline. The sample proves joins and field availability; it does not estimate market efficiency or trader skill. See [data definitions](../docs/data_dictionary.md) and [findings](../docs/data_feasibility.md).

In [1]:
from pathlib import Path
import csv, json
root = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
run_id = "20260909T153138473505Z"
clean = root / "data" / "clean" / run_id
raw = root / "data" / "raw" / run_id
def table(name):
    with (clean / name).open(encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))
markets = table("markets.csv")
trades = table("trades.csv")
wallet_categories = table("wallet_category.csv")
report = json.loads((clean / "quality_report.json").read_text(encoding="utf-8"))
print({"markets": len(markets), "trade_rows": len(trades), "wallets": report["distinct_sample_wallets"]})

{'markets': 12, 'trade_rows': 1178, 'wallets': 418}


## Market, resolution, and raw trade

Resolution labels require explicit resolved status and a unique 0/1 outcome-price vector. BUY/SELL and the purchased or sold outcome are separate fields.

In [2]:
market = next(m for m in markets if m["category"] == "Politics")
trade = next(t for t in trades if t["market_id"] == market["market_id"])
print(json.dumps({"market": {k: market[k] for k in ["question", "category", "resolution", "forecast_outcome", "final_outcome"]},
                  "trade": {k: trade[k] for k in ["wallet", "timestamp_utc", "side", "outcome", "price", "size_shares"]}}, indent=2))
source_rows = json.loads((raw / trade["source_file"]).read_text(encoding="utf-8"))
source = source_rows[int(trade["source_row"])]
assert source["conditionId"] == trade["condition_id"]
assert float(source["price"]) == float(trade["price"])
print("Raw-to-clean row trace verified.")

{
  "market": {
    "question": "Will AfD win 50 or more seats in the 2026 Sachsen-Anhalt parliamentary elections?",
    "category": "Politics",
    "resolution": "No",
    "forecast_outcome": "Yes",
    "final_outcome": "0"
  },
  "trade": {
    "wallet": "0x99f7df5c91f048d291c3759cd086ce670196f5ae",
    "timestamp_utc": "2026-09-06T22:37:04+00:00",
    "side": "BUY",
    "outcome": "Yes",
    "price": "0.001",
    "size_shares": "5.0"
  }
}
Raw-to-clean row trace verified.


## Coverage and missingness

Horizon prices refer to the first outcome token and use closedTime as a provisional anchor. Missing values are not imputed. These markets were selected by tag and recency, with a small older-listing stratum.

In [3]:
print(json.dumps({"price_coverage": report["price_coverage"],
                  "trade_stop_reasons": report["trade_stop_reasons"],
                  "missing_raw_categories": report["raw_category_missing"],
                  "overlapping_domain_markets": report["multiple_domain_markets"],
                  "duplicate_candidates_retained": report["duplicate_candidates_retained"]}, indent=2))
assert len({m["condition_id"] for m in markets}) == len(markets)
assert not report["rejected_rows"]
for m in markets:
    for days in (30, 7, 1):
        if m[f"price_{days}d_status"] == "available":
            assert 0 <= float(m[f"price_{days}_days_before"]) <= 1
            assert 0 <= float(m[f"price_{days}d_age_seconds"]) <= 21600

{
  "price_coverage": {
    "30": {
      "market_not_open": 11,
      "available": 1
    },
    "7": {
      "market_not_open": 10,
      "available": 2
    },
    "1": {
      "market_not_open": 7,
      "available": 5
    }
  },
  "trade_stop_reasons": {
    "short_page": 8,
    "sample_cap": 4
  },
  "missing_raw_categories": 12,
  "overlapping_domain_markets": 3,
  "duplicate_candidates_retained": {
    "trades_*_*.json": 5,
    "wallet_activity_*.json": 0
  }
}


## One wallet across domain tags

The displayed counts are sampled trade records, not independent bets. Combined categories preserve tag overlap. Profit, ROI, and win rate cannot be inferred from this bounded activity sample.

In [4]:
for row in wallet_categories:
    print(f"{row['category']}: {row['sample_num_trades']} trades across {row['sample_num_markets']} markets")
print("Activity types:", report["wallet_activity_types"])
print("Joined wallet TRADE rows:", report["wallet_activity_trade_rows_joined"])
assert sum(int(r["sample_num_trades"]) for r in wallet_categories) == report["wallet_activity_trade_rows_joined"]
assert all(r["profit"] == r["roi"] == r["win_rate"] == "" for r in wallet_categories)

Business;Technology: 2 trades across 1 markets
Climate: 2 trades across 1 markets
Crypto: 30 trades across 19 markets
Economy;Finance: 1 trades across 1 markets
Economy;Politics: 1 trades across 1 markets
Geopolitics: 4 trades across 2 markets
Geopolitics;Politics: 1 trades across 1 markets
Politics: 111 trades across 7 markets
Sports: 36 trades across 7 markets
Activity types: {'TRADE': 188, 'REWARD': 3, 'REDEEM': 8, 'MAKER_REBATE': 1}
Joined wallet TRADE rows: 188


## Next collection decisions

Choose markets with sufficient pre-event history; define a reviewable category taxonomy; test bounded time-window pagination against independent totals; reconstruct wallet inventory and cash flows before estimating performance. Separate event information time from settlement and closure times. Only then expand to the course EDA figures and modeling questions.